# B0 → B5: безопасный роутер по патчам

Отдельный эксперимент: лёгкая SegFormer-B0 остаётся неизменной, а тяжёлой моделью становится SegFormer-B5 — самый сильный Cityscapes checkpoint, уже предусмотренный в `core.models`. B5 запускается только на выбранных патчах. Обучается лишь роутер на Cityscapes P0, P1 остаётся честной проверкой.

Ноутбук сохраняет каждый готовый патч B5 в Google Drive: повторный запуск не тратит уже использованное GPU-время.

In [ ]:
from pathlib import Path
from google.colab import drive
MOUNT = Path('/content/gdrive')
if not (MOUNT / 'MyDrive').exists():
    drive.mount(str(MOUNT))
%pip -q install --upgrade-strategy only-if-needed gdown transformers cityscapesscripts pyarrow


In [ ]:
# Подготовка B0 и данных. Результаты прошлого B0→B2 запуска используются повторно.
import os, shutil, subprocess, sys, zipfile
import gdown

assert __import__('torch').cuda.is_available(), 'Включите T4 GPU: Runtime → Change runtime type.'
REPO = Path('/content/error-geometry-routing')
DRIVE_DIR = MOUNT / 'MyDrive' / 'error-geometry-routing'
OLD_RESULTS = DRIVE_DIR / 'posthoc_router_20261027' / 'results'
OUT = DRIVE_DIR / 'b5_safe_router_20261027'
OUT.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'experiments/frozen-router-results-20261025',
                    'https://github.com/Ottobiss/error-geometry-routing.git', str(REPO)], check=True)
os.chdir(REPO)
if OLD_RESULTS.exists():
    shutil.copytree(OLD_RESULTS, REPO / 'results', dirs_exist_ok=True)

download = Path('/content/cityscapes-archives'); download.mkdir(exist_ok=True)
archives = {'val.zip': '1aErjLEt25oJEX48W-zxHwITEvNAeKxGJ',
            'gtFine_trainvaltest.zip': '1AxGkcacU_SZXPSjxUtzKgT3XSZM032bk'}
for name, file_id in archives.items():
    path = download / name
    if not path.exists() or path.stat().st_size == 0:
        gdown.download(id=file_id, output=str(path), quiet=False)

data = REPO / 'data' / 'cityscapes'; data.mkdir(parents=True, exist_ok=True)
for name in archives:
    with zipfile.ZipFile(download / name) as zf:
        for item in zf.infolist():
            target = data / item.filename
            if not target.exists(): zf.extract(item, data)
canonical = data / 'leftImg8bit' / 'val'
if not list(canonical.rglob('*_leftImg8bit.png')):
    for src in data.rglob('*_leftImg8bit.png'):
        dst = canonical / src.parent.name / src.name; dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists(): shutil.move(str(src), str(dst))

def run(*args): subprocess.run([sys.executable, '-m', *args], cwd=REPO, check=True)
if not (REPO / 'results/cityscapes/components.parquet').exists(): run('features.components', '--dataset', 'cityscapes')
if not (REPO / 'results/cityscapes/logits_b0.npy').exists(): run('features.logits', '--dataset', 'cityscapes')
assert len(list(canonical.rglob('*_leftImg8bit.png'))) == 500
print('B0 и Cityscapes готовы.')


In [ ]:
# B5 на окнах 256×256 с контекстом 64. Полный B5-кадр намеренно не запускаем:
# для каскада он не нужен и на T4 часто не помещается в память.
import json, time
import numpy as np
import torch
from core.data import DATASETS, get_dataset, loader
from core.models import load_model
from core.paths import DatasetResults
from core.tiles import grid, window

# Первый прогон: MARGIN=64. Для проверки контекста измените только на 128;
# имена результатов автоматически станут другими и m=64 сохранится.
TILE, MARGIN, BATCH = 256, 64, 1
cfg = f'b5_image_t{TILE}_m{MARGIN}'
TAG = f'b5_t{TILE}_m{MARGIN}'
res = DatasetResults('cityscapes')
tile_store = OUT / f'b5_tile_masks_t{TILE}_m{MARGIN}'; tile_store.mkdir(exist_ok=True)
ds = get_dataset('cityscapes', full=True)
boxes = grid(ds.size, TILE)
b5 = load_model('b5')

@torch.inference_mode()
def predict_tiles(src):
    scale = src.shape[2] / ds.size[0]; ws = round((TILE + 2 * MARGIN) * scale)
    answer = np.zeros((ds.size[1], ds.size[0]), np.uint8)
    for x0, y0, x1, y1 in boxes:
        wx, wy = window((x0, y0, x1, y1), TILE, MARGIN, ds.size)
        sx = min(round(wx * scale), src.shape[2] - ws); sy = min(round(wy * scale), src.shape[1] - ws)
        crop = src[:, sy:sy + ws, sx:sx + ws][None]
        pred = b5(crop, (TILE + 2 * MARGIN, TILE + 2 * MARGIN)).argmax(1)[0].cpu().numpy()
        answer[y0:y1, x0:x1] = pred[y0-wy:y1-wy, x0-wx:x1-wx]
    return answer

t0 = time.time(); ids, masks = [], []
for i, sample in enumerate(loader(ds), 1):
    image_id = sample['image_id']; path = tile_store / f'{image_id}.npy'
    if path.exists(): pred = np.load(path)
    else:
        pred = predict_tiles(sample['image'].cuda()); np.save(path, pred)
    ids.append(image_id); masks.append(pred)
    if i % 25 == 0 or i == len(ds): print(f'{i}/{len(ds)}; {time.time()-t0:.0f} s; сохранено {path.name}')
np.savez_compressed(res.tiles(cfg), image_id=np.array(ids), tiled=np.stack(masks))
res.tiles(cfg, 'json').write_text(json.dumps({'heavy': 'b5', 'tile': TILE, 'margin': MARGIN,
    'input': 'image', 'n_tiles': len(boxes), 'note': 'B5 only on windows; per-image masks are checkpointed on Drive'}, ensure_ascii=False))
print('B5-ответы по патчам:', res.tiles(cfg))


In [ ]:
# Обучение роутера B0→B5. P1 не участвует в fit и выборе alpha.
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from core.components import IGNORE, NC
from core.evaluation import entropy_score, routing_cache
from core.tiles import grid, tile_effects, tile_index
from features.logits import LogitStore

info = json.loads(res.tiles(cfg, 'json').read_text()); z = np.load(res.tiles(cfg))
tile_pos = {x: i for i, x in enumerate(z['image_id'])}
boxes, tidx = grid(ds.size, TILE), tile_index(ds.size, TILE); n_tiles = len(boxes)
area = np.bincount(tidx.ravel(), minlength=n_tiles)
logits = LogitStore('cityscapes'); rows = []
for population in ('p0', 'p1'):
    cache = routing_cache('cityscapes', population)
    for image_id in cache['image_id']:
        k = cache['pos'][image_id]; gt, b0 = cache['gt'][k], cache['b0'][k]; b5t = z['tiled'][tile_pos[image_id]]
        valid = gt != IGNORE; _, _, delta = tile_effects(gt, b0, b5t, tidx, n_tiles)
        prob = logits[image_id].softmax(0); ent = entropy_score(prob).cpu().numpy(); top = prob.topk(2, dim=0).values.cpu().numpy()
        def mean(a): return np.bincount(tidx.ravel(), weights=a.ravel(), minlength=n_tiles) / area
        edge = np.zeros(b0.shape, bool); edge[:, :-1] |= b0[:, :-1] != b0[:, 1:]; edge[:-1, :] |= b0[:-1, :] != b0[1:, :]
        overlap = np.bincount(cache['comp'][k].ravel().astype(np.int64)*n_tiles+tidx.ravel(), minlength=(cache['comp'][k].max()+1)*n_tiles).reshape(-1,n_tiles)
        cls = np.bincount(tidx.ravel()*NC+b0.ravel(), minlength=n_tiles*NC).reshape(n_tiles,NC)
        fields = {'m_entropy': mean(ent), 'ent_mean': mean(ent), 'ent_frac': mean(ent > .5),
                  'ent_p90': np.array([np.quantile(ent[y0:y1,x0:x1],.9) for x0,y0,x1,y1 in boxes]),
                  'margin_mean': mean(1-(top[0]-top[1])), 'pmax_mean': mean(top[0]),
                  'boundary': mean(edge), 'n_comp': (overlap>0).sum(0)}
        fields.update({f'cls_{c}': cls[:,c]/area for c in range(NC)})
        rows.append(pd.DataFrame({'image_id': image_id, 'population': population, 'tile': np.arange(n_tiles), **fields,
          'delta': delta, 'n_valid': np.bincount(tidx.ravel(), weights=valid.ravel(), minlength=n_tiles),
          'b0_error': np.bincount(tidx.ravel(), weights=(valid&(b0!=gt)).ravel(), minlength=n_tiles),
          'b5_error': np.bincount(tidx.ravel(), weights=(valid&(b5t!=gt)).ravel(), minlength=n_tiles)}))
table = pd.concat(rows, ignore_index=True); table.to_parquet(OUT/f'{TAG}_tile_features.parquet', index=False)
# Это отвечает на главный диагностический вопрос: B5 реально чаще исправляет B0 или нет?
pairwise = pd.DataFrame([{'population': pop,
    'B5_better_tile_fraction': (g.b5_error < g.b0_error).mean(),
    'B5_worse_tile_fraction': (g.b5_error > g.b0_error).mean(),
    'same_error_tile_fraction': (g.b5_error == g.b0_error).mean(),
    'mean_B0_errors_per_tile': g.b0_error.mean(),
    'mean_B5_errors_per_tile': g.b5_error.mean()}
    for pop, g in table.groupby('population')])
pairwise.to_csv(OUT/f'{TAG}_tile_diagnostics.csv', index=False); print(pairwise.to_string(index=False))
train = table.query("population == 'p0'").reset_index(drop=True); test = table.query("population == 'p1'").reset_index(drop=True)
features = ['m_entropy','ent_mean','ent_p90','ent_frac','margin_mean','pmax_mean','boundary','n_comp'] + [f'cls_{c}' for c in range(NC)]
def fit_gain(frame):
    d=frame[frame.n_valid>0]; m=HistGradientBoostingRegressor(learning_rate=.05,max_iter=120,max_leaf_nodes=7,min_samples_leaf=200,l2_regularization=1.,random_state=0)
    return m.fit(d[features], d.delta/d.n_valid, sample_weight=d.n_valid)
def frame_rank(frame, value):
    out=np.empty(len(frame))
    for _, ix in frame.groupby('image_id',sort=False).indices.items(): out[ix]=rankdata(value[ix])/len(ix)
    return out
def gain_per_frame(frame, value):
    ans=[]
    for _, ix in frame.groupby('image_id',sort=False).indices.items():
        d=frame.iloc[ix].delta.to_numpy(); ans.append(np.mean([d[np.argsort(value[ix])[-k:]].sum() for k in (1,2,3)]))
    return np.asarray(ans)
oof=np.empty(len(train)); gkf=GroupKFold(n_splits=5)
for tr, va in gkf.split(train, groups=train.image_id): oof[va]=fit_gain(train.iloc[tr]).predict(train.iloc[va][features])
er, gr = frame_rank(train,train.m_entropy.to_numpy()), frame_rank(train,oof); base=gain_per_frame(train,train.m_entropy.to_numpy())
choice=[]
for a in (0,.025,.05,.1,.2,.35,.5):
    diff=gain_per_frame(train,(1-a)*er+a*gr)-base; se=diff.std(ddof=1)/np.sqrt(len(diff)); choice.append({'alpha':a,'oof_delta_pixels':diff.mean(),'se':se,'passes_guard':a>0 and diff.mean()>1.96*se})
choice=pd.DataFrame(choice); ok=choice[choice.passes_guard]; alpha=0. if ok.empty else float(ok.sort_values('oof_delta_pixels',ascending=False).iloc[0].alpha)
b5_gain=fit_gain(train).predict(test[features]); safe=(1-alpha)*frame_rank(test,test.m_entropy.to_numpy())+alpha*frame_rank(test,b5_gain)
scores={'entropy':test.m_entropy.to_numpy(), f'safe-blend-b5(alpha={alpha:g})':safe}
out=test[['image_id','tile','delta','n_valid','b0_error','b5_error']].copy()
for name,value in scores.items(): out[name]=value
diagnostic=[]; target=(out.delta>0).to_numpy(); density=out.delta/out.n_valid.clip(lower=1)
for name,value in scores.items(): diagnostic.append({'method':name,'AUROC':roc_auc_score(target,value),'AP':average_precision_score(target,value),'Spearman':spearmanr(value,density).statistic})
choice.to_csv(OUT/f'{TAG}_alpha_selection_p0.csv',index=False); out.to_parquet(OUT/f'{TAG}_p1_scores.parquet',index=False)
print(f'alpha={alpha:g}; 0 означает: B5-контекст не прошёл защиту на P0.')
pd.DataFrame(diagnostic).sort_values('Spearman',ascending=False), choice


In [ ]:
# Итог: mIoU и paired bootstrap на P1. Одинаковый бюджет B5-патчей для каждого метода.
from core.evaluation import _miou, _miou_batch
from routing.tiles import select
cache = routing_cache('cityscapes','p1'); frames=list(test.groupby('image_id',sort=True).indices.items()); effects={}
for image_id,_ in frames:
    k=cache['pos'][image_id]; effects[image_id]=tile_effects(cache['gt'][k],cache['b0'][k],z['tiled'][tile_pos[image_id]],tidx,n_tiles)[:2]
rng=np.random.default_rng(0); weights=rng.multinomial(len(frames),np.full(len(frames),1/len(frames)),size=2000); rows=[]
for budget in (1,2,3):
    confs={name:np.stack([(effects[i][1]+effects[i][0][select(score[ix],k=budget)].sum(0)).ravel() for i,ix in frames]) for name,score in scores.items()}
    ref=confs['entropy']; ref_miou=_miou(ref.sum(0).reshape(19,19)); ref_boot=_miou_batch((weights@ref).reshape(-1,19,19))
    for name,conf in confs.items():
        diff=_miou_batch((weights@conf).reshape(-1,19,19))-ref_boot
        rows.append({'B5_patches_per_frame':budget,'method':name,'mIoU':_miou(conf.sum(0).reshape(19,19)), 'delta_to_entropy':_miou(conf.sum(0).reshape(19,19))-ref_miou, 'ci_lo':np.percentile(diff,2.5),'ci_hi':np.percentile(diff,97.5)})
report=pd.DataFrame(rows).sort_values(['B5_patches_per_frame','mIoU'],ascending=[True,False]); report.to_csv(OUT/f'{TAG}_p1_miou_bootstrap.csv',index=False); report


## Консервативное pixel-level fusion B0 + B5

Следующие две ячейки реализуют пункт 3. Entropy по-прежнему выбирает патч, поэтому бюджет B5 не меняется. Но внутри выбранного патча gate может оставить исходную метку B0, если B5 с высокой вероятностью не исправляет её. Gate обучается только на конфликтующих B0/B5 пикселях P0; порог выбирается по out-of-fold результату P0.

In [ ]:
# Строим gate: принять B5 только когда он вероятно исправляет именно B0.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

PIXEL_CAP_PER_FRAME = 1000  # ограничивает RAM; выборка равномерна по кадрам, не по числу пикселей
rng = np.random.default_rng(42)
def boundary_map(pred):
    edge = np.zeros(pred.shape, bool)
    edge[:, :-1] |= pred[:, :-1] != pred[:, 1:]
    edge[:-1, :] |= pred[:-1, :] != pred[1:, :]
    return edge

def gate_features(image_id, b0, b5, pick):
    # B0 logits — глобальная ветка; class B5 — локальное уточнение.
    prob = logits[image_id].softmax(0).cpu().numpy()
    top = np.partition(prob, -2, axis=0)[-2:]
    entropy = -(prob * np.log(np.maximum(prob, 1e-8))).sum(0)
    h, w = b0.shape; yy, xx = np.indices((h, w))
    scalar = np.column_stack([prob.max(0).ravel()[pick], entropy.ravel()[pick],
        (top[1]-top[0]).ravel()[pick], boundary_map(b0).ravel()[pick],
        (xx.ravel()[pick]/max(w-1,1)), (yy.ravel()[pick]/max(h-1,1))]).astype(np.float32)
    eye = np.eye(NC, dtype=np.float32)
    return np.concatenate([scalar, eye[b0.ravel()[pick]], eye[b5.ravel()[pick]]], axis=1)

# Мишень +1: B5 права, B0 нет. −1: B0 права, B5 нет. Остальные пиксели не дают решения gate.
xs, ys, groups = [], [], []
for population in ('p0',):
    cache0 = routing_cache('cityscapes', population)
    for image_id in cache0['image_id']:
        k = cache0['pos'][image_id]; gt, b0 = cache0['gt'][k], cache0['b0'][k]; b5 = z['tiled'][tile_pos[image_id]]
        # Берём только пиксели с ненулевым выигрышем: одна модель права, другая — нет.
        conflict = (gt != IGNORE) & ((b0 == gt) != (b5 == gt))
        pos = np.flatnonzero(conflict.ravel())
        if len(pos) == 0: continue
        if len(pos) > PIXEL_CAP_PER_FRAME: pos = rng.choice(pos, PIXEL_CAP_PER_FRAME, replace=False)
        xs.append(gate_features(image_id, b0, b5, pos))
        ys.append((b5.ravel()[pos] == gt.ravel()[pos]).astype(np.int8))
        groups.append(np.full(len(pos), image_id))
X = np.concatenate(xs); y = np.concatenate(ys); group = np.concatenate(groups)
print('Обучающих конфликтующих пикселей:', len(y), '; доля, где B5 права:', y.mean())

def make_gate():
    return HistGradientBoostingClassifier(learning_rate=.06, max_iter=120, max_leaf_nodes=15,
        min_samples_leaf=100, l2_regularization=2., random_state=0)
oof = np.empty(len(y))
for tr, va in GroupKFold(n_splits=5).split(X, y, groups=group):
    m = make_gate().fit(X[tr], y[tr]); oof[va] = m.predict_proba(X[va])[:, 1]
# Порог допуска выбирается по парному net gain на независимых от fit кадрах P0.
signed = 2*y - 1
gate_rows = []
for threshold in (.50, .60, .70, .80, .90, .95, .98):
    frame_gain = []
    for image_id in np.unique(group):
        q = group == image_id; frame_gain.append((signed[q] * (oof[q] >= threshold)).sum())
    frame_gain = np.asarray(frame_gain); mean = frame_gain.mean(); se = frame_gain.std(ddof=1)/np.sqrt(len(frame_gain))
    gate_rows.append({'threshold':threshold, 'oof_net_fixed_pixels_per_frame':mean, 'se':se,
        'passes_95pct_guard': mean > 1.96*se})
gate_selection = pd.DataFrame(gate_rows); allowed = gate_selection.query('passes_95pct_guard')
threshold = 1.1 if allowed.empty else float(allowed.sort_values('oof_net_fixed_pixels_per_frame', ascending=False).iloc[0].threshold)
gate = make_gate().fit(X, y)
gate_selection.to_csv(OUT/f'{TAG}_pixel_gate_oof_selection.csv', index=False)
print(f'Выбран threshold={threshold:g}; 1.1 означает: gate не доказал безопасную замену B0 на P0.')
gate_selection


In [ ]:
# Применяем gate только внутри трёх наиболее энтропийных патчей каждого P1-кадра
# и сравниваем с обычной полной заменой B0→B5 на тех же патчах.
from core.evaluation import _miou, _miou_batch
from routing.tiles import select
cache1 = routing_cache('cityscapes', 'p1')
frames = list(test.groupby('image_id', sort=True).indices.items())
entropy = scores['entropy']; base_effects, fused_effects, accepted = {}, {}, []
for image_id, ix in frames:
    k = cache1['pos'][image_id]; gt, b0 = cache1['gt'][k], cache1['b0'][k]; b5 = z['tiled'][tile_pos[image_id]]
    allowed_tiles = select(entropy[ix], k=3)
    # В P1 не используем GT: во время инференса доступны только B0, B5 и номера выбранных патчей.
    candidate = (b0 != b5) & np.isin(tidx, allowed_tiles)
    pos = np.flatnonzero(candidate.ravel()); take = np.zeros(b0.size, bool)
    if len(pos) and threshold <= 1:
        take[pos] = gate.predict_proba(gate_features(image_id, b0, b5, pos))[:,1] >= threshold
    take = take.reshape(b0.shape); fused = b0.copy(); fused[take] = b5[take]
    base_effects[image_id] = tile_effects(gt, b0, b5, tidx, n_tiles)[:2]
    fused_effects[image_id] = tile_effects(gt, b0, fused, tidx, n_tiles)[:2]
    accepted.append({'image_id':image_id, 'accepted_pixels_top3':int(take.sum()), 'candidate_pixels_top3':int(len(pos))})
pd.DataFrame(accepted).to_csv(OUT/f'{TAG}_p1_pixel_gate_acceptance.csv', index=False)

rng=np.random.default_rng(0); weights=rng.multinomial(len(frames), np.full(len(frames),1/len(frames)), size=2000); rows=[]
for budget in (1,2,3):
    base=np.stack([(base_effects[i][1]+base_effects[i][0][select(entropy[ix],k=budget)].sum(0)).ravel() for i,ix in frames])
    fused=np.stack([(fused_effects[i][1]+fused_effects[i][0][select(entropy[ix],k=budget)].sum(0)).ravel() for i,ix in frames])
    ref_miou=_miou(base.sum(0).reshape(19,19)); ref_boot=_miou_batch((weights@base).reshape(-1,19,19))
    for name, conf in [('entropy: replace whole B5 patch', base), ('entropy + pixel gate B0/B5', fused)]:
        diff=_miou_batch((weights@conf).reshape(-1,19,19))-ref_boot
        rows.append({'B5_patches_per_frame':budget, 'method':name, 'mIoU':_miou(conf.sum(0).reshape(19,19)),
          'delta_to_whole_patch_entropy':_miou(conf.sum(0).reshape(19,19))-ref_miou,
          'ci_lo':np.percentile(diff,2.5), 'ci_hi':np.percentile(diff,97.5)})
pixel_gate_report=pd.DataFrame(rows).sort_values(['B5_patches_per_frame','mIoU'],ascending=[True,False])
pixel_gate_report.to_csv(OUT/f'{TAG}_p1_pixel_gate_miou_bootstrap.csv',index=False)
pixel_gate_report
